In [2]:
##########################################################################
# 인스타 그램의 해쉬태그 수집하기 - by 서진수 (수정 및 DB 연동 통합판)
##########################################################################
#Step 1. 필요한 모듈과 라이브러리를 로딩합니다.
from bs4 import BeautifulSoup
import time
import math
import os
import random
import unicodedata   # 인스타그램의 해시태그 수집 중 자음/모음 분리현상 방지용 모듈
import urllib.request
import urllib.parse  # 📌 튕김 방지 URL 변환용
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import pandas as pd
import pyautogui
import pymysql
import re
import sys
import getpass       # 🔒 비밀번호 입력을 화면에 숨기기 위한 모듈 추가

# 비트맵 이미지 아이콘을 위한 대체 딕셔너리를 만든다
bmp_map = dict.fromkeys(range(0x10000, sys.maxunicode + 1), 0xfffd)

print("=" *80)
print(" 인스타그램 해시태그 크롤링 프로그램")
print("=" *80)
print("\n")

#Step 2. 사용자에게 필요한 정보들을를 입력 받습니다.
v_id = input('1.인스타그램 아이디(이메일 등)를 입력하세요: ')
v_passwd = getpass.getpass('2.인스타그램 비밀번호를 입력하세요(입력시 화면에 안보임): ')

query_txt = input("3.검색할 해쉬태그를 입력하세요(예: 강남맛집): ").replace('#', '').strip()
cnt = int(input('4.크롤링 할 게시글 건수는 몇건입니까?: '))
real_cnt = math.ceil(cnt / 10)
comment_cnt = int(input('5.수집할 댓글은 한 게시글당 몇 건입니까?(예: 5): '))

f_dir=input('6.파일이 저장될 경로만 쓰세요(기본경로 : c:\\py_temp\\ ) : ')
if f_dir =='' :
    f_dir = "c:\\py_temp\\"

#Step 3. 결과를 저장할 폴더명과 파일명을 설정하고 폴더를 생성합니다.
s_time = time.time( )
now = time.localtime()
s = '%04d-%02d-%02d-%02d-%02d-%02d' % (now.tm_year, now.tm_mon, now.tm_mday, now.tm_hour, now.tm_min, now.tm_sec)

os.makedirs(f_dir + s + '-' + query_txt, exist_ok=True)
os.chdir(f_dir + s + '-' + query_txt)
ff_name = f_dir + s + '-' + query_txt + '\\' + s + '-' + query_txt + '.txt'
fc_name = f_dir + s + '-' + query_txt + '\\' + s + '-' + query_txt + '.csv'
fx_name = f_dir + s + '-' + query_txt + '\\' + s + '-' + query_txt + '.xls'

# Step 4. 인스타그램 자동 로그인 하기 및 드라이버 셋팅
options = Options()
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option("useAutomationExtension", False)
options.add_argument("--disable-blink-features=AutomationControlled")

try:
    s_srv = Service("c:/py_temp/chromedriver.exe")
    driver = webdriver.Chrome(service=s_srv, options=options)
except:
    driver = webdriver.Chrome(options=options)

driver.get("https://www.instagram.com/")
time.sleep(random.randrange(1,5))

driver.maximize_window( )

print("\n요청하신 데이터를 추출중이오니 잠시만 기다려 주세요~~~~^^\n")

wait = WebDriverWait(driver, 10)

try:
    # ID와 비번 입력후 로그인하기 
    try:
        eid = wait.until(EC.element_to_be_clickable((By.NAME, 'username')))
        for a in v_id :
            eid.send_keys(a)
            time.sleep(0.3)
            
        epwd = driver.find_element(By.NAME,'password')
        for b in v_passwd :
            epwd.send_keys(b)
            time.sleep(0.5)
            
        epwd.send_keys(Keys.ENTER)
        time.sleep(random.randrange(1,5))
    except:
        # 서진수님 원본 방식을 그대로 적용 (이메일/패스워드 이름 기반 폼 탐색)
        eid = driver.find_element(By.NAME,'email')
        for a in v_id :
            eid.send_keys(a)
            time.sleep(0.3)
        epwd = driver.find_element(By.NAME,'pass')
        for b in v_passwd :
            epwd.send_keys(b)
            time.sleep(0.5)
        driver.find_element(By.XPATH,'//*[@id="login_form"]/div/div[1]/div/div[3]/div/div/div').click()
        time.sleep(random.randrange(1,5))

    # ==========================================================
    # Step 5. 검색할 해쉬태그 입력하기 (팝업 선 처리 + 원본 검색)
    # ==========================================================
    
    print("팝업('나중에 하기')을 확인하고 있습니다...")

    # 1. '로그인 정보 저장' 팝업 끄기
    try:
        btn_save = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, "//*[text()='나중에 하기']"))
        )
        btn_save.click()
        time.sleep(3)
    except:
        pass

    # 2. '알림 설정' 팝업 끄기
    try:
        btn_noti = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, "//button[text()='나중에 하기']"))
        )
        btn_noti.click()
        time.sleep(3)
    except:
        pass

    # 3. '사용하지 않음' 팝업 끄기
    try:
        btn_not_use = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, "//*[text()='사용하지 않음']"))
        )
        btn_not_use.click()
        time.sleep(3)
    except:
        pass

    # 왼쪽의 검색버튼 클릭
    element = wait.until(EC.element_to_be_clickable((By.XPATH,"//a[.//svg[@aria-label='검색' or @aria-label='Search'] or .//span[text()='검색' or text()='Search']]")))
    element.click()
    time.sleep(2)

    # 검색할 키워드 입력하기
    selector = (
        "input[type='text'][placeholder='검색'],"
        "input[type='text'][aria-label*='검색'],"
        "[role='textbox'][aria-label*='검색']"
    )
    search_box = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, selector)))

    for c in query_txt:
        search_box.send_keys(c)
        time.sleep(0.2)

    time.sleep(3)
    
    # 어설프게 클릭을 시도하다가 뒤에 있는 홈 화면의 다른 해시태그를 누르는 사고 방지
    encoded_query = urllib.parse.quote(query_txt)
    driver.get(f"https://www.instagram.com/explore/tags/{encoded_query}/")
    time.sleep(6)

except Exception as e:
    print("\n[접속 에러 발생] 로그인 및 검색 도중 문제가 생겼습니다:", e)


# ==========================================================
# Step 6. 첫 번째 게시글 클릭 후 순회 크롤링 및 5가지 데이터 추출
# ==========================================================
post_no = []              # 엑셀 출력을 위한 전체 행 번호 (1, 2, 3...)
post_board_no = []        # 게시글 고유 번호 (1, 1, 1, 2, 2...)
post_authors = []   
post_contents = []          
post_comments = []     
post_likes = []           
post_hashtags = []     

total_count = 0  # 크롤링한 게시글 수 카운트
row_count = 0    # 분리된 개별 행(Row) 수 카운트

print('\n게시글 본문과 댓글 수집을 시작합니다. 화면을 닫지 말아주세요!')

try:
    first_post = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a[href^="/p/"]')))
    first_post.click()
    time.sleep(3)
except Exception as e:
    print("첫번째 게시물을 찾지 못했습니다. 크롤링을 종료합니다.", e)
    total_count = cnt 

while total_count < cnt:
    total_count += 1
    print(f"\n🚀 [진행상황] 총 {cnt}건 중 {total_count}번째 게시글 수집 중 =========")
    
    f = open(ff_name, 'a', encoding='UTF-8')
    f.write("\n")
    f.write(f"[{total_count} 번째 게시글 정보]====\n")

    # 💡 [추가/수정됨] 무한 스크롤 및 "댓글 더 읽어들이기" 제어 (목표치 미달시 종료 로직 포함)
    scroll_attempts = 0
    no_new_comments_count = 0  # 새 댓글이 안 생기는 루프 연속 카운트
    prev_comment_count = 0     # 이전 루프의 댓글 갯수 기록
    
    while scroll_attempts < 30: # 최대 30번 스크롤 시도 (넉넉한 버퍼)
        temp_html = driver.page_source
        temp_soup = BeautifulSoup(temp_html, 'html.parser')
        
        # 본문 내용 추출 (댓글 리스트에서 본문을 제외해야 정확한 댓글 수를 파악)
        try:
            t_content_node = temp_soup.select_one('h1._ap3a._aaco._aacu._aacx._aad7._aade')
            t_content_text = t_content_node.get_text(separator=' ', strip=True) if t_content_node else ""
        except:
            t_content_text = ""
            
        # 댓글 컨테이너 안의 댓글들을 파악
        t_comment_nodes = temp_soup.select('div.xt0psk2 span[dir="auto"]')
        current_valid_comments = []
        for temp_c in t_comment_nodes:
            t_c_text = temp_c.get_text(separator=' ', strip=True)
            if t_c_text and t_c_text not in t_content_text:
                current_valid_comments.append(t_c_text)
                
        current_comment_count = len(current_valid_comments)
        
        # 이전 개수와 변동이 있는지 검사 (댓글이 더이상 새로고침 되지 않는지 판단)
        if current_comment_count == prev_comment_count:
            no_new_comments_count += 1
        else:
            no_new_comments_count = 0  # 댓글이 늘어났으면 카운트 초기화
            
        prev_comment_count = current_comment_count

        if scroll_attempts > 0:
            print(f"   💬 현재 로드된 댓글 수: {current_comment_count} 개 (목표: {comment_cnt} 개)")
        
        # 1. 목표한 댓글 갯수에 도달했으면 중단!
        if current_comment_count >= comment_cnt:
            print("   ✅ 목표 댓글 수에 도달했습니다!")
            break
            
        # 2. 버튼 클릭이나 스크롤을 여러번 시도했는데도 댓글 갯수가 3번(루프 연속)이나 같다면, 모든 댓글이 불러와진 것으로 간주하고 강제 탈출!
        if no_new_comments_count >= 3:
            print("   ⚠️ 더 이상 불러올 수 있는 댓글이 없습니다. 발견된 댓글까지만 수집하고 마칩니다.")
            break
            
        try:
            # 먼저 마지막 댓글 요소로 스크롤하여 오토 페이징(다음 댓글 불러오기) 유도
            DOM_comments = driver.find_elements(By.CSS_SELECTOR, 'div.xt0psk2 span[dir="auto"]')
            if DOM_comments:
                driver.execute_script("arguments[0].scrollIntoView({block: 'center', behavior: 'smooth'});", DOM_comments[-1])
            time.sleep(1.0)
            
            # 알려주신 동그라미 모양의 (+) "댓글 더 읽어들이기" 기호 요소의 정밀 탐색 (svg 파일과 클래스 매칭)
            more_btns = driver.find_elements(By.CSS_SELECTOR, "div._abm0 svg[aria-label='댓글 더 읽어들이기'], div._abm0 svg[aria-label='더 보기']")
            if more_btns:
                # 일반 클릭(click())이 먹히지 않을 것을 대비한 네이티브 이벤트 디스패치 전송
                driver.execute_script("""
                    var target = arguments[0].closest('div._abm0') || arguments[0];
                    target.dispatchEvent(new MouseEvent('click', {bubbles: true, cancelable: true, view: window}));
                """, more_btns[0])
                time.sleep(1.5)
            else:
                # 위 아이콘이 안 보일 경우 보조 장치인 '댓글 더 보기' 텍스트 클릭 시도
                text_more_btns = driver.find_elements(By.XPATH, "//*[text()='댓글 더 보기' or text()='답글 더 보기']")
                if text_more_btns:
                    driver.execute_script("arguments[0].click();", text_more_btns[0])
                    time.sleep(1.5)
        except Exception as e:
            time.sleep(1.5)
            pass
            
        scroll_attempts += 1


    # 모든 댓글을 최대로 로딩한 뒤 최종 돔(DOM) 객체를 완성
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    # [데이터 최신화 추출]
    
    # 1. 작성자
    try:
        author_node = soup.select_one('span.x1lliihq.x1plvlek.xryxfnj.x1n2onr6.xyejjpt.x15dsfln.x193iq5w.xeuugli.x1fj9vlw.x13faqbe.x1vvkbs.x1s928wv.xhkezso.x1gmr53x.x1cpjm7i.x1fgarty.x1943h6x.x1i0vuye.xvs91rp.x1s688f.x5n08af.x10wh9bi.xpm28yp.x8viiok.x1o7cslx')
        if not author_node:
            author_node = soup.select_one('header h2 span[dir="auto"], h2 a')
        author = author_node.get_text(strip=True) if author_node else "작성자 없음"
    except:
        author = "작성자 없음"
    
    # 2. 본문 내용
    try:
        content_node = soup.select_one('h1._ap3a._aaco._aacu._aacx._aad7._aade')
        content = content_node.get_text(separator=' ', strip=True) if content_node else "내용 없음"
    except:
        content = "내용 없음"

    # 3. 댓글 (수집된 컨테이너 데이터 재반환)
    try:
        comment_nodes = soup.select('div.xt0psk2 span[dir="auto"]')
        comments_texts = []
        for c in comment_nodes:
            c_text = c.get_text(separator=' ', strip=True)
            if c_text and c_text not in content: 
                comments_texts.append(c_text)
                if len(comments_texts) >= comment_cnt: # 목표치 이상 담지 않음
                    break
    except:
        comments_texts = []

    # 4. 좋아요 수
    try:
        like_nodes = soup.select('span.xdj266r.x14z9mp.xat24cr.x1lziwak.xexx8yu.xyri2b.x18d9i69.x1c1uobl.x1hl2dhg.x16tdsg8.x1vvkbs')
        likes = "좋아요 없음"
        for node in like_nodes:
            text = node.get_text(strip=True)
            if text.replace(',', '').isdigit():
                likes = text
                break
        
        if likes == "좋아요 없음":
            fallback = soup.find(string=re.compile(r'좋아요[\s]*[\d,]+개|여러 명'))
            if fallback:
                likes = fallback.strip()
    except:
        likes = "좋아요 없음"
        
    # 5. 해시태그
    try:
        hashtag_nodes = soup.select('a[href*="/explore/tags/"]')
        hashtags_list = []
        for h in hashtag_nodes:
            tag_text = h.get_text(strip=True)
            if tag_text.startswith('#') and tag_text not in hashtags_list:
                hashtags_list.append(tag_text)
        hashtags = " ".join(hashtags_list) if hashtags_list else "해시태그 없음"
    except:
        hashtags = "해시태그 없음"

    # [파일 및 데이터 배열에 각각 저장]
    f.write("1.작성자: " + author + "\n")
    f.write("2.본문 내용: " + content + "\n")
    
    if not comments_texts:
        # 댓글이 1개도 없을 경우
        f.write("3.댓글: 댓글 없음\n")
        
        row_count += 1
        post_no.append(row_count)
        post_board_no.append(total_count)
        post_authors.append(author)
        post_contents.append(content)
        post_comments.append("댓글 없음")
        post_likes.append(likes)
        post_hashtags.append(hashtags)
    else:
        # 추출된 댓글을 번호순으로 차례대로 삽입 (저장하는 것도 실제 뽑은 개수만큼만)
        for idx, c_text in enumerate(comments_texts):
            f.write(f"3.댓글({idx+1}): {c_text}\n")
            
            row_count += 1
            post_no.append(row_count)
            post_board_no.append(total_count)
            post_authors.append(author)
            post_contents.append(content)
            post_comments.append(c_text)
            post_likes.append(likes)
            post_hashtags.append(hashtags)
            
    f.write("4.좋아요 수: " + likes + "\n")
    f.write("5.해쉬태그: " + hashtags + "\n")

    f.close()
    
    if total_count >= cnt:
        print(f"\n✅ 수집 목표량({cnt}게시물) 달성 완료!")
        break

    # 다음 게시물로 넘어가기 시도
    try:
        next_btn = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'svg[aria-label="다음"]')))
        driver.execute_script("arguments[0].closest('button').click();", next_btn)
        time.sleep(random.uniform(1.8, 3.2))
    except Exception as e:
        try:
             webdriver.ActionChains(driver).send_keys(Keys.ARROW_RIGHT).perform()
             time.sleep(random.uniform(1.8, 3.2))
        except:
             print("다음 게시물 화살표를 찾을 수 없습니다. (종료)")
             break


# ==========================================================
# Step 7. DataFrame 생성 후 xls와 csv 형태로 저장하기
# ==========================================================
insta_df = pd.DataFrame()
insta_df['번호'] = pd.Series(post_no)
insta_df['게시글 번호'] = pd.Series(post_board_no)
insta_df['작성자'] = pd.Series(post_authors)
insta_df['본문 내용'] = pd.Series(post_contents)
insta_df['댓글'] = pd.Series(post_comments)
insta_df['좋아요 수'] = pd.Series(post_likes)
insta_df['해쉬태그'] = pd.Series(post_hashtags)

try:
    insta_df.to_csv(fc_name, encoding="utf-8-sig", index=False)
    insta_df.to_excel(fx_name, index=False, engine='openpyxl')
except Exception as e:
    print("\n경고: 엑셀 또는 CSV 파일 저장에 실패했습니다. (openpyxl 등 확인 필요)", e)


# ==========================================================
# Step 8. 수집한 수집정보 MySQL DB에 넣기
# ==========================================================
try:
    conn = pymysql.connect(
        host='localhost',         
        user='root',              
        password='Jx03151616~~',  
        db='youtube_db',  
        charset='utf8mb4',        
        cursorclass=pymysql.cursors.DictCursor
    )
    with conn.cursor() as cursor:
        
        # [주의] 이 줄을 사용할지 여부를 결정해 주세요.
        # 코드 실행시마다 이전 DB 데이터를 전부 지우고 엑셀과 동일한 최신 데이터만 남기고 싶다면 앞의 #을 풀어주시면 됩니다.
        # cursor.execute("DROP TABLE IF EXISTS insta_posts")
        
        create_table_sql = """
        CREATE TABLE IF NOT EXISTS insta_posts (
            id INT AUTO_INCREMENT PRIMARY KEY, 
            post_no INT,                 -- 엑셀의 '번호'
            post_board_no INT,           -- 엑셀의 '게시글 번호'
            author VARCHAR(255),         
            content TEXT,                 
            comments TEXT,
            likes VARCHAR(50),
            hashtags TEXT
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4;
        """
        cursor.execute(create_table_sql)
        
        insert_sql = """
        INSERT INTO insta_posts (post_no, post_board_no, author, content, comments, likes, hashtags) 
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        """
        for i in range(len(post_authors)):
            cursor.execute(insert_sql, (
                post_no[i],              
                post_board_no[i],        
                post_authors[i], 
                post_contents[i], 
                post_comments[i],
                post_likes[i],
                post_hashtags[i]
            ))
        conn.commit()
        print(f"🎉 짝짝짝! 인스타그램 DB 저장까지 완벽하게 완료되었습니다!")
        
except Exception as e:
    print(f"\n[DB 에러] DB 저장 중 에러가 발생했습니다: {e}")
finally:
    try:
        conn.close()
    except:
        pass


# ==========================================================
# Step 9. 요약 정보 출력하기
# ==========================================================
e_time = time.time( )
t_time = e_time - s_time

print("\n")
print("=" *120)
print(f"1.모든 작업 종료. 수집된 전체(인스타그램) 게시글 수는 {total_count} 건 입니다.")
print(f"  * 게시글 및 분할된 댓글을 포함한 총 수집 행(Row) 개수는 {row_count}개 입니다.")
print("2.총 소요시간은 %s 초 입니다 " %round(t_time,1))
print("3.파일 저장 완료: txt 파일명 : %s " %ff_name)
print("4.파일 저장 완료: csv 파일명 : %s " %fc_name)
print("5.파일 저장 완료: xls 파일명 : %s " %fx_name)
print("=" *120)

driver.quit() # 안전하게 창 닫기


 인스타그램 해시태그 크롤링 프로그램




ValueError: invalid literal for int() with base 10: ''